# EXP-04 (`CLOUD-LANE-01`): Live `vLLM 0.15+` + `CowrieKVConnectorV1` Benchmark & `SupportsHMA` Proof

**Goal:** Run an end-to-end live GPU verification of `CowrieKVConnectorV1` on `vLLM 0.15+` (V1 engine) across a **full two-process restart** (`Phase 1: Cold Store` -> complete VRAM teardown -> `Phase 2: Warm Cowrie Restore`), proving:

1. `vllm.general_plugins` / `KVConnectorFactory` registers `CowrieKVConnectorV1`.
2. **`SupportsHMA` is genuinely `True` at runtime** (both statically on the factory-resolved class via real `vllm...SupportsHMA` and dynamically on the live Scheduler/Worker instances).
3. Real multi-layer KV blocks are saved to `0600` HMAC-SHA256 + 4 MiB Merkle `.state.cowrie` shards on disk and restored into a fresh OS process with **100% exact greedy token parity**.
4. Measurable **tokens/s**, **TTFT**, **KV cache hit rate**, and **`CowrieKVConnectorV1`'s own counters** — failing loudly if the connector is silently bypassed.

### Hardware Profiles & Credit Pool
- **Credit pool:** Google AI Ultra — **Google Colab Compute Units** (`2,000 CU/mo`, 90-day rollover). **Zero paid GCP spend, zero external accounts, no gated HF token required** for the default profiles.
- **Primary profile:** **NVIDIA H100 80GB HBM3 (`sm_90`)** (~18 CU/hr). Default model: `Qwen/Qwen3-32B` (BF16, ~64 GB weights). Switch to `MODEL_PROFILE = "70b-awq"` (`Qwen/Qwen2.5-72B-Instruct-AWQ`, ungated) or `"70b-fp8"` (`nvidia/Llama-3.3-70B-Instruct-FP8`) for 70B-class runs on H100 80GB.
- **Blackwell variant (supported):** **NVIDIA RTX PRO 6000 Blackwell 96GB (`sm_120`, G4)** (~8.9 CU/hr). Runs both `qwen3-32b` and `70b-awq` with extra KV headroom; `CowrieKVConnectorV1` natively supports both FlashAttention `(2, num_blocks, block_size, ...)` and FlashInfer `(num_blocks, 2, block_size, ...)` KV layouts.

> **Instructions:** Select an H100 80GB (or RTX PRO 6000 Blackwell) runtime in Colab Ultra and click **Runtime -> Run all**. No manual edits are required.

In [ ]:
import json
import os
import secrets
from pathlib import Path

# ==============================================================================
# Experiment Configuration (zero manual edits required for H100 80GB default)
# ==============================================================================
MODEL_PROFILE = os.environ.get("EXP04_MODEL_PROFILE", "qwen3-32b")

MODEL_PROFILES = {
    # Default: Qwen3-32B BF16 (~64 GB weights) -> fits H100 80GB and RTX PRO 6000 Blackwell 96GB; ungated.
    "qwen3-32b": {
        "model_id": "Qwen/Qwen3-32B",
        "dtype": "bfloat16",
        "quantization": None,
        "max_model_len": 8192,
        "gpu_memory_utilization": 0.90,
        "min_vram_gib": 74.0,
        "notes": "Default H100 80GB / Blackwell 96GB profile (ungated BF16).",
    },
    # Higher KV-headroom 32B option on H100 / Blackwell (~34 GB weights); ungated.
    "qwen3-32b-fp8": {
        "model_id": "Qwen/Qwen3-32B-FP8",
        "dtype": "auto",
        "quantization": None,
        "max_model_len": 16384,
        "gpu_memory_utilization": 0.88,
        "min_vram_gib": 42.0,
        "notes": "FP8 32B variant for high KV headroom on H100 80GB / Blackwell 96GB (ungated).",
    },
    # 70B switch for H100 80GB / Blackwell 96GB (ungated INT4-AWQ, ~41 GB weights).
    "70b-awq": {
        "model_id": "Qwen/Qwen2.5-72B-Instruct-AWQ",
        "dtype": "float16",
        "quantization": None,
        "max_model_len": 8192,
        "gpu_memory_utilization": 0.88,
        "min_vram_gib": 48.0,
        "notes": "70B-class ungated switch for H100 80GB / Blackwell 96GB (~41 GB AWQ weights).",
    },
    # 70B FP8 switch for H100 80GB / Blackwell 96GB (~71 GB weights; set HF_TOKEN if gated).
    "70b-fp8": {
        "model_id": "nvidia/Llama-3.3-70B-Instruct-FP8",
        "dtype": "auto",
        "quantization": None,
        "max_model_len": 4096,
        "gpu_memory_utilization": 0.92,
        "min_vram_gib": 74.0,
        "notes": "70B native FP8 switch for H100 80GB / Blackwell 96GB (may require HF_TOKEN).",
    },
    # Optional smaller smoke check if testing on a 24-48 GB GPU.
    "qwen3-8b-smoke": {
        "model_id": "Qwen/Qwen3-8B",
        "dtype": "bfloat16",
        "quantization": None,
        "max_model_len": 8192,
        "gpu_memory_utilization": 0.85,
        "min_vram_gib": 20.0,
        "notes": "Optional smaller smoke profile (not the primary 32B/70B benchmark).",
    },
}

assert MODEL_PROFILE in MODEL_PROFILES, (
    f"Unknown EXP04_MODEL_PROFILE={MODEL_PROFILE!r}; choose from {sorted(MODEL_PROFILES)}"
)
PROFILE = MODEL_PROFILES[MODEL_PROFILE]

COWRIE_REPO_URL = os.environ.get("COWRIE_REPO_URL", "https://github.com/Neumenon/cowrie.git")
COWRIE_GIT_REF = os.environ.get("COWRIE_GIT_REF", "feat/cloud-lane-01")
STATE_DIR = Path(os.environ.get("EXP04_STATE_DIR", "/tmp/cowrie_exp04_kv"))
RECEIPT_PATH = Path(os.environ.get("EXP04_RECEIPT_PATH", "/tmp/cowrie_exp04_receipt.json"))

# ~120 telemetry rows yields ~2,400-3,000 prompt tokens (~600-750 MiB KV shard on Qwen3-32B).
PROMPT_ROWS = int(os.environ.get("EXP04_PROMPT_ROWS", "120"))
DECODE_MAX_TOKENS = int(os.environ.get("EXP04_DECODE_TOKENS", "64"))
# Eager mode keeps 2-process startup fast & uniform across Hopper sm_90 and Blackwell sm_120.
# Set EXP04_ENFORCE_EAGER=0 to benchmark with torch.compile / CUDA graphs.
ENFORCE_EAGER = os.environ.get("EXP04_ENFORCE_EAGER", "1") == "1"

# Qwen3-32B (64L) and 72B (80L) KV shards exceed the 512 MiB default inflight cap at >=2K tokens.
COWRIE_MAX_INFLIGHT_BYTES = 4 * 1024 * 1024 * 1024  # 4 GiB host-RAM staging cap
COWRIE_MAX_CACHE_BYTES = 32 * 1024 * 1024 * 1024    # 32 GiB local disk cache cap

# Ephemeral per-run 256-bit HMAC key (no external secrets required).
COWRIE_HMAC_KEY_HEX = os.environ.get("COWRIE_HMAC_KEY", secrets.token_hex(32))

# Set True for unattended runs to automatically disconnect Colab runtime at the end.
AUTO_UNASSIGN_COLAB_RUNTIME = os.environ.get("EXP04_AUTO_UNASSIGN", "0") == "1"

CONFIG = {
    "experiment": "EXP-04",
    "lane": "CLOUD-LANE-01",
    "model_profile": MODEL_PROFILE,
    "model_id": PROFILE["model_id"],
    "dtype": PROFILE["dtype"],
    "quantization": PROFILE["quantization"],
    "max_model_len": PROFILE["max_model_len"],
    "gpu_memory_utilization": PROFILE["gpu_memory_utilization"],
    "min_vram_gib": PROFILE["min_vram_gib"],
    "prompt_rows": PROMPT_ROWS,
    "decode_max_tokens": DECODE_MAX_TOKENS,
    "enforce_eager": ENFORCE_EAGER,
    "cowrie_max_inflight_bytes": COWRIE_MAX_INFLIGHT_BYTES,
    "cowrie_max_cache_bytes": COWRIE_MAX_CACHE_BYTES,
    "state_dir": str(STATE_DIR),
    "receipt_path": str(RECEIPT_PATH),
    "cowrie_repo_url": COWRIE_REPO_URL,
    "cowrie_git_ref": COWRIE_GIT_REF,
    "auto_unassign_colab_runtime": AUTO_UNASSIGN_COLAB_RUNTIME,
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
def make_corpus_prefix(corpus_id: str, rows: int) -> str:
    """Build a deterministic, distinct long structured prefix so prompts do not collide in VRAM."""
    lines = [
        f"corpus={corpus_id} node=gpu-{i:03d} rack=r{i % 8} "
        f"temp_c={51 + ((i * 7 + len(corpus_id)) % 23)} "
        f"util_pct={72 + ((i * 5) % 27)} mem_gib={61 + (i % 18)} status=healthy"
        for i in range(rows)
    ]
    return (
        f"You are an infrastructure diagnostics assistant reviewing verified telemetry batch {corpus_id}.\n"
        + "\n".join(lines)
        + "\n\n"
    )


def build_benchmark_prompts(rows: int) -> dict[str, str]:
    """Return three disjoint prefix corpora for TTFT, Decode+Parity, and New-Suffix reuse."""
    prefix_ttft = make_corpus_prefix("EXP04_TTFT_ALPHA", rows)
    prefix_decode = make_corpus_prefix("EXP04_DECODE_BETA", rows)
    prefix_suffix = make_corpus_prefix("EXP04_SUFFIX_GAMMA", rows)
    return {
        "prompt_ttft": (
            prefix_ttft
            + "Question: Which node index is the final entry in batch EXP04_TTFT_ALPHA?\nAnswer:"
        ),
        "prompt_decode": (
            prefix_decode
            + "Question: Summarize the cluster telemetry trends across racks r0 through r7, "
            "noting temperature spread, utilization range, and the final node index.\nAnswer:"
        ),
        # Phase 1 seeds the exact block-aligned prefix of prefix_suffix;
        # Phase 2 queries prefix_suffix with a brand-new suffix question.
        "prompt_prefix_seed": prefix_suffix,
        "prompt_new_suffix": (
            prefix_suffix
            + "Question: Identify the highest rack index and summarize overall node health.\nAnswer:"
        ),
    }


def assert_connector_was_used(store_res: dict, restore_res: dict, hma_proof: dict) -> None:
    """Fail loudly if SupportsHMA is false or CowrieKVConnectorV1 was silently bypassed."""
    errors: list[str] = []

    if not hma_proof.get("supports_hma_native_import"):
        errors.append("SupportsHMA was NOT imported natively from vllm (fallback stub active).")
    if not hma_proof.get("issubclass_supports_hma"):
        errors.append("issubclass(CowrieKVConnectorV1, SupportsHMA) is False.")
    if not hma_proof.get("has_request_finished_all_groups"):
        errors.append("CowrieKVConnectorV1 missing request_finished_all_groups.")

    st = store_res.get("connector_stats", {}).get("totals", {})
    rt = restore_res.get("connector_stats", {}).get("totals", {})

    # Phase 1 (Store) assertions
    if st.get("instance_count", 0) < 1:
        errors.append("Phase 1: zero CowrieKVConnectorV1 instances wrote stats (connector not initialized).")
    if not st.get("all_supports_hma", False):
        errors.append(f"Phase 1: live connector supports_hma check failed (totals={st}).")
    if st.get("save_kv_layer_calls", 0) <= 0:
        errors.append("Phase 1: save_kv_layer_calls == 0 (worker save hook never executed).")
    if st.get("saves_completed", 0) < 3:
        errors.append(
            f"Phase 1: expected >= 3 saved .state.cowrie shards, got saves_completed={st.get('saves_completed')}."
        )
    if st.get("saved_bytes", 0) <= 0 or st.get("saved_tokens", 0) <= 0:
        errors.append(
            f"Phase 1: zero saved KV payload (saved_tokens={st.get('saved_tokens')}, saved_bytes={st.get('saved_bytes')})."
        )
    if st.get("saves_failed", 0) > 0:
        errors.append(f"Phase 1: saves_failed={st.get('saves_failed')} (StatePage persistence errors).")
    if st.get("saves_skipped_inflight_cap", 0) > 0:
        errors.append(
            f"Phase 1: saves_skipped_inflight_cap={st.get('saves_skipped_inflight_cap')} "
            "(increase COWRIE_MAX_INFLIGHT_BYTES)."
        )
    if store_res.get("verified_shards_count", 0) < 3:
        errors.append(
            f"Phase 1: expected >= 3 hash-verified .state.cowrie shards on disk, "
            f"got {store_res.get('verified_shards_count')}."
        )

    # Phase 2 (Restore) assertions
    if rt.get("instance_count", 0) < 1:
        errors.append("Phase 2: zero CowrieKVConnectorV1 instances wrote stats (connector not initialized).")
    if not rt.get("all_supports_hma", False):
        errors.append(f"Phase 2: live connector supports_hma check failed (totals={rt}).")
    if rt.get("get_num_new_matched_tokens_calls", 0) <= 0:
        errors.append("Phase 2: get_num_new_matched_tokens_calls == 0 (scheduler hook never executed).")
    if rt.get("scheduler_hits", 0) < 3:
        errors.append(
            f"Phase 2: expected >= 3 Cowrie scheduler prefix hits across restart, got {rt.get('scheduler_hits')}."
        )
    if rt.get("start_load_kv_calls", 0) <= 0 or rt.get("loads_completed", 0) < 3:
        errors.append(
            f"Phase 2: Cowrie disk KV restore did not execute for all 3 prompts "
            f"(start_load_kv_calls={rt.get('start_load_kv_calls')}, loads_completed={rt.get('loads_completed')})."
        )
    if rt.get("loaded_tokens", 0) <= 0 or rt.get("loaded_bytes", 0) <= 0:
        errors.append(
            f"Phase 2: zero KV payload loaded (loaded_tokens={rt.get('loaded_tokens')}, loaded_bytes={rt.get('loaded_bytes')})."
        )
    if rt.get("loads_failed", 0) > 0:
        errors.append(f"Phase 2: loads_failed={rt.get('loads_failed')} (StatePage restore errors).")

    # Greedy token parity assertion
    if store_res.get("decode_token_ids") != restore_res.get("decode_token_ids"):
        errors.append(
            "Greedy token parity FAILED across restart: "
            f"store[:8]={store_res.get('decode_token_ids', [])[:8]} vs "
            f"restore[:8]={restore_res.get('decode_token_ids', [])[:8]}"
        )

    if errors:
        raise RuntimeError(
            "EXP-04 FAIL-LOUD CHECK FAILED (connector bypassed, broken, or parity mismatch):\n  - "
            + "\n  - ".join(errors)
        )


def compute_benchmark_summary(store_res: dict, restore_res: dict, hma_proof: dict, gpu_info: dict) -> dict:
    """Compute tokens/s, TTFT, KV cache hit rate, and full connector counter summary."""
    rt = restore_res["connector_stats"]["totals"]
    st = store_res["connector_stats"]["totals"]

    cold_ttft_ms = float(store_res["ttft_ms"])
    warm_ttft_ms = float(restore_res["ttft_ms"])
    ttft_speedup_x = round(cold_ttft_ms / max(warm_ttft_ms, 1e-3), 3)

    cold_decode_tokens = int(store_res["decode_output_tokens"])
    warm_decode_tokens = int(restore_res["decode_output_tokens"])
    cold_e2e_s = max(float(store_res["decode_e2e_ms"]) / 1000.0, 1e-6)
    warm_e2e_s = max(float(restore_res["decode_e2e_ms"]) / 1000.0, 1e-6)
    suffix_tokens = int(restore_res["new_suffix_output_tokens"])
    suffix_e2e_s = max(float(restore_res["new_suffix_e2e_ms"]) / 1000.0, 1e-6)

    # Estimate post-TTFT decode rate using the matched-length warm TTFT probe on the same GPU/process.
    warm_decode_only_s = max(warm_e2e_s - (warm_ttft_ms / 1000.0), 1e-6)
    warm_decode_only_tok_s = round(max(warm_decode_tokens - 1, 1) / warm_decode_only_s, 2)

    restore_prompt_tokens_total = (
        int(restore_res["ttft_prompt_tokens"])
        + int(restore_res["decode_prompt_tokens"])
        + int(restore_res["new_suffix_prompt_tokens"])
    )
    loaded_tokens = int(rt.get("loaded_tokens", 0))
    matched_tokens = int(rt.get("scheduler_matched_tokens", 0))
    kv_hit_rate_loaded_pct = round(100.0 * loaded_tokens / max(restore_prompt_tokens_total, 1), 2)
    kv_hit_rate_scheduler_pct = round(100.0 * matched_tokens / max(restore_prompt_tokens_total, 1), 2)
    request_hit_rate_pct = round(100.0 * min(int(rt.get("scheduler_hits", 0)), 3) / 3.0, 2)

    vllm_reported_cached = restore_res.get("vllm_num_cached_tokens_total")

    return {
        "experiment": "EXP-04",
        "lane": "CLOUD-LANE-01",
        "hardware": gpu_info,
        "config": CONFIG,
        "hma_proof": hma_proof,
        "metrics": {
            "prompt_tokens_ttft": int(restore_res["ttft_prompt_tokens"]),
            "prompt_tokens_decode": int(restore_res["decode_prompt_tokens"]),
            "prompt_tokens_new_suffix": int(restore_res["new_suffix_prompt_tokens"]),
            "restore_prompt_tokens_total": restore_prompt_tokens_total,
            "cold_ttft_ms": round(cold_ttft_ms, 2),
            "warm_cowrie_ttft_ms": round(warm_ttft_ms, 2),
            "ttft_speedup_x": ttft_speedup_x,
            "cold_decode_e2e_ms": round(float(store_res["decode_e2e_ms"]), 2),
            "warm_decode_e2e_ms": round(float(restore_res["decode_e2e_ms"]), 2),
            "cold_e2e_tokens_per_s": round(cold_decode_tokens / cold_e2e_s, 2),
            "warm_e2e_tokens_per_s": round(warm_decode_tokens / warm_e2e_s, 2),
            "warm_decode_only_tokens_per_s": warm_decode_only_tok_s,
            "new_suffix_warm_e2e_ms": round(float(restore_res["new_suffix_e2e_ms"]), 2),
            "new_suffix_warm_e2e_tokens_per_s": round(suffix_tokens / suffix_e2e_s, 2),
            "kv_cache_hit_rate_loaded_pct": kv_hit_rate_loaded_pct,
            "kv_cache_hit_rate_scheduler_pct": kv_hit_rate_scheduler_pct,
            "kv_cache_request_hit_rate_pct": request_hit_rate_pct,
            "vllm_num_cached_tokens_total": vllm_reported_cached,
            "greedy_token_parity": store_res["decode_token_ids"] == restore_res["decode_token_ids"],
            "decode_output_tokens": warm_decode_tokens,
            "state_shards_count": store_res["verified_shards_count"],
            "state_shards_total_bytes": store_res["state_bytes_total"],
            "hmac_sha256_verified": store_res["all_shards_hmac_verified"],
        },
        "connector_counters": {
            "phase_1_store": store_res["connector_stats"],
            "phase_2_restore": restore_res["connector_stats"],
        },
        "verdict": "PASS" if warm_ttft_ms < cold_ttft_ms else "FUNCTIONAL_PASS_NO_TTFT_SPEEDUP",
    }


PROMPTS = build_benchmark_prompts(PROMPT_ROWS)
print(
    "Prepared 4 benchmark prompt strings:",
    {k: len(v) for k, v in PROMPTS.items()},
)

## 1. GPU Preflight & Dependency Installation (Colab Ultra)

- Verifies an NVIDIA GPU is attached and has sufficient VRAM for `MODEL_PROFILE`.
- Installs `vllm>=0.15.0` (if not already present) and builds/installs `cowrie-ai` directly from this repository (installing a minimal Rust toolchain via `rustup` if `cargo` is absent).
- Uses zero paid GCP resources and zero external cloud credentials.

In [ ]:
import shutil
import subprocess
import sys

if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No NVIDIA GPU detected (nvidia-smi not found). "
        "Attach an H100 80GB or RTX PRO 6000 Blackwell runtime in Colab Ultra."
    )

smi_out = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version,compute_cap,power.limit",
        "--format=csv,noheader,nounits",
    ],
    text=True,
).strip()
first_gpu = [x.strip() for x in smi_out.splitlines()[0].split(",")]
gpu_name, mem_mib_s, driver_ver, compute_cap, power_limit_w = first_gpu
vram_gib = round(float(mem_mib_s) / 1024.0, 2)

if "H100" in gpu_name.upper():
    detected_profile = "H100_80GB_HOPPER"
elif "PRO 6000" in gpu_name.upper() or compute_cap.startswith("12."):
    detected_profile = "RTX_PRO_6000_BLACKWELL"
else:
    detected_profile = f"OTHER_CUDA_SM_{compute_cap.replace('.', '')}"

GPU_INFO = {
    "gpu_name": gpu_name,
    "vram_gib": vram_gib,
    "driver_version": driver_ver,
    "compute_capability": compute_cap,
    "power_limit_w": power_limit_w,
    "detected_profile": detected_profile,
}
print("GPU Preflight:", json.dumps(GPU_INFO, indent=2))

if vram_gib < PROFILE["min_vram_gib"]:
    raise RuntimeError(
        f"Attached GPU '{gpu_name}' has {vram_gib} GiB VRAM, below the {PROFILE['min_vram_gib']} GiB "
        f"minimum for MODEL_PROFILE={MODEL_PROFILE!r} ({PROFILE['model_id']}). "
        f"Switch Colab runtime to H100 80GB / RTX PRO 6000 Blackwell 96GB, or choose a smaller MODEL_PROFILE."
    )

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import shutil
import subprocess
import sys


def _run(cmd, env=None, cwd=None):
    print("+", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=cwd)


def _version_tuple(v: str) -> tuple[int, ...]:
    parts = []
    for tok in v.split("."):
        digits = "".join(ch for ch in tok if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)


# 1. Ensure vLLM 0.15+ is installed
need_vllm = True
try:
    cur_vllm = importlib.metadata.version("vllm")
    if _version_tuple(cur_vllm) >= (0, 15, 0):
        print(f"vLLM already installed: {cur_vllm}")
        need_vllm = False
    else:
        print(f"Upgrading vLLM from {cur_vllm} to >=0.15.0...")
except importlib.metadata.PackageNotFoundError:
    print("Installing vllm>=0.15.0...")

if need_vllm:
    _run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "vllm>=0.15.0"])

# 2. Ensure minimal Rust cargo toolchain is available to compile libcowrie_core.so
cargo_bin_dir = Path.home() / ".cargo" / "bin"
if cargo_bin_dir.is_dir() and str(cargo_bin_dir) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{cargo_bin_dir}:{os.environ.get('PATH', '')}"

if not shutil.which("cargo"):
    print("Rust cargo not found; installing minimal rustup toolchain...")
    _run([
        "bash",
        "-lc",
        "curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal --default-toolchain stable",
    ])
    os.environ["PATH"] = f"{cargo_bin_dir}:{os.environ.get('PATH', '')}"

assert shutil.which("cargo"), "cargo still missing after rustup installation!"

# 3. Locate local Cowrie checkout or clone from COWRIE_REPO_URL
cowrie_repo_dir = None
for cand in (Path.cwd(), Path.cwd().parent, Path("/content/cowrie")):
    if (cand / "Cargo.toml").is_file() and (cand / "adapters" / "vllm" / "cowrie_kv_connector.py").is_file():
        cowrie_repo_dir = cand.resolve()
        break

if cowrie_repo_dir is None:
    cowrie_repo_dir = Path("/content/cowrie") if Path("/content").is_dir() else Path("/tmp/cowrie-src")
    if cowrie_repo_dir.exists():
        shutil.rmtree(cowrie_repo_dir)
    try:
        _run(["git", "clone", "--depth", "1", "--branch", COWRIE_GIT_REF, COWRIE_REPO_URL, str(cowrie_repo_dir)])
    except subprocess.CalledProcessError:
        print(f"Branch {COWRIE_GIT_REF!r} not found on remote; falling back to main...")
        if cowrie_repo_dir.exists():
            shutil.rmtree(cowrie_repo_dir)
        _run(["git", "clone", "--depth", "1", "--branch", "main", COWRIE_REPO_URL, str(cowrie_repo_dir)])

COWRIE_GIT_SHA = subprocess.check_output(
    ["git", "-C", str(cowrie_repo_dir), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Using Cowrie repo at {cowrie_repo_dir} (HEAD {COWRIE_GIT_SHA})")

# 4. Install cowrie-ai Python package without touching installed torch/vllm deps
_run([sys.executable, "-m", "pip", "install", "-q", "setuptools>=68", "wheel"])
build_env = os.environ.copy()
build_env["CARGO_TARGET_DIR"] = "/tmp/cowrie-cargo-target"
_run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "--no-deps",
        "--force-reinstall",
        str(cowrie_repo_dir),
    ],
    env=build_env,
)

# 5. Verify native Cowrie library + vllm.general_plugins entry point
import cowrie

eps = [ep.name for ep in importlib.metadata.entry_points(group="vllm.general_plugins")]
print("Installed cowrie-ai:", importlib.metadata.version("cowrie-ai"))
print("Cowrie SHA-NI hardware accel:", bool(cowrie.has_shani()))
print("vllm.general_plugins entry points:", eps)
assert "cowrie_kv_connector" in eps, (
    f"cowrie_kv_connector entry point missing from vllm.general_plugins: {eps}"
)

## 2. Register `CowrieKVConnectorV1` & Prove `SupportsHMA` at Runtime

This cell resolves `CowrieKVConnectorV1` **through vLLM's own `KVConnectorFactory`** (testing `vllm.general_plugins` auto-discovery first) and asserts that it genuinely subclasses vLLM's real `SupportsHMA` mixin — never the fallback stub.

In [ ]:
import importlib.metadata
import json
import vllm
from vllm.config import KVTransferConfig
from vllm.distributed.kv_transfer.kv_connector.factory import KVConnectorFactory
from vllm.distributed.kv_transfer.kv_connector.v1.base import (
    KVConnectorBase_V1,
    SupportsHMA,
)

vllm_ver = getattr(vllm, "__version__", importlib.metadata.version("vllm"))
ver_parts = tuple(int("".join(c for c in p if c.isdigit()) or "0") for p in vllm_ver.split(".")[:3])
assert ver_parts >= (0, 15, 0), f"Expected vLLM >= 0.15.0, got {vllm_ver}"

# Trigger vLLM's real general plugin discovery
try:
    from vllm.plugins import load_general_plugins
    load_general_plugins()
except Exception as exc:
    print(f"Note: load_general_plugins raised {exc!r}; checking explicit registration...")

registry = getattr(KVConnectorFactory, "_registry", {})
auto_registered = "CowrieKVConnectorV1" in registry
if not auto_registered:
    from cowrie_vllm_adapters._plugin import register_vllm_plugin
    register_vllm_plugin()

assert "CowrieKVConnectorV1" in getattr(KVConnectorFactory, "_registry", {}), (
    "CowrieKVConnectorV1 failed to register in KVConnectorFactory._registry!"
)

# Resolve through vLLM's factory (exact path used by EngineCore)
kv_probe_cfg = KVTransferConfig(kv_connector="CowrieKVConnectorV1", kv_role="kv_both")
resolved_cls = KVConnectorFactory.get_connector_class(kv_probe_cfg)

from cowrie_vllm_adapters.cowrie_kv_connector import (
    CowrieKVConnectorV1,
    SUPPORTS_HMA_NATIVE,
    read_connector_stats,
    reset_connector_stats,
)

assert resolved_cls is CowrieKVConnectorV1, f"Unexpected factory class: {resolved_cls}"
assert issubclass(resolved_cls, KVConnectorBase_V1), "Not a KVConnectorBase_V1 subclass!"
assert SUPPORTS_HMA_NATIVE is True, (
    "SUPPORTS_HMA_NATIVE is False — CowrieKVConnectorV1 fell back to stub _SupportsHMA!"
)
assert issubclass(resolved_cls, SupportsHMA), (
    f"Runtime SupportsHMA check FAILED! MRO={[c.__name__ for c in resolved_cls.__mro__]}"
)
has_all_groups = callable(getattr(resolved_cls, "request_finished_all_groups", None))
assert has_all_groups, "request_finished_all_groups method missing on CowrieKVConnectorV1!"

HMA_PROOF = {
    "vllm_version": vllm_ver,
    "cowrie_git_sha": COWRIE_GIT_SHA,
    "auto_registered_via_entry_point": auto_registered,
    "factory_resolved_class": f"{resolved_cls.__module__}.{resolved_cls.__qualname__}",
    "mro": [f"{c.__module__}.{c.__qualname__}" for c in resolved_cls.__mro__],
    "supports_hma_class": f"{SupportsHMA.__module__}.{SupportsHMA.__qualname__}",
    "supports_hma_native_import": SUPPORTS_HMA_NATIVE,
    "issubclass_supports_hma": issubclass(resolved_cls, SupportsHMA),
    "has_request_finished_all_groups": has_all_groups,
}
print("SupportsHMA RUNTIME PROOF: PASS")
print(json.dumps(HMA_PROOF, indent=2))

## 3. Two-Process Cross-Restart KV Offload & Decode Benchmark

We write a standalone worker script `/tmp/cowrie_exp04_worker.py` and invoke it in **two completely separate OS processes**:

1. **Process 1 (`--phase store`):** Cold GPU VRAM, empty `STATE_DIR`. Measures **Cold TTFT** (`max_tokens=1`), **Cold Decode** (`DECODE_MAX_TOKENS` greedy tokens + parity reference), and seeds a 3rd disjoint prefix corpus. Persists 3 HMAC-authenticated `.state.cowrie` shards on disk and verifies every shard with `cowrie.verify_statepage_file(..., verify_hash=True)`. Process 1 then exits completely, freeing 100% of GPU VRAM.
2. **Process 2 (`--phase restore`):** Fresh OS process, 0% in-GPU prefix cache, same `STATE_DIR`. Restores all 3 prefixes from `.state.cowrie` shards on disk — including a **brand-new suffix question** on the 3rd corpus — and measures **Warm Cowrie TTFT**, **Decode tokens/s**, **KV cache hit rate**, **greedy token parity**, and **connector counters**.

In [ ]:
from pathlib import Path

WORKER_SCRIPT_PATH = Path("/tmp/cowrie_exp04_worker.py")

WORKER_SOURCE = r'''#!/usr/bin/env python3
"""Standalone single-phase worker for EXP-04 (invoked in separate OS processes for store vs restore)."""
from __future__ import annotations

import argparse
import gc
import json
import os
from pathlib import Path
import time


def _extract_metrics_optional(out) -> dict:
    m = getattr(out, "metrics", None)
    info = {
        "num_cached_tokens": getattr(out, "num_cached_tokens", None),
        "vllm_ttft_ms": None,
    }
    if m is not None:
        arr = getattr(m, "arrival_time", None)
        ft = getattr(m, "first_token_time", None)
        if arr is not None and ft is not None and ft >= arr:
            info["vllm_ttft_ms"] = round((ft - arr) * 1000.0, 2)
    return info


def _shutdown_llm(llm) -> None:
    try:
        eng = getattr(llm, "llm_engine", None)
        ec = getattr(eng, "engine_core", None)
        if ec is not None and hasattr(ec, "shutdown"):
            ec.shutdown()
    except Exception:
        pass
    del llm
    gc.collect()


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--phase", choices=["store", "restore"], required=True)
    parser.add_argument("--config-json", required=True)
    parser.add_argument("--prompts-json", required=True)
    parser.add_argument("--result-json", required=True)
    args = parser.parse_args()

    os.environ["VLLM_USE_V1"] = "1"
    os.environ["VLLM_NO_USAGE_STATS"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    cfg = json.loads(Path(args.config_json).read_text(encoding="utf-8"))
    prompts = json.loads(Path(args.prompts_json).read_text(encoding="utf-8"))
    state_dir = Path(cfg["state_dir"])
    state_dir.mkdir(mode=0o700, parents=True, exist_ok=True)

    import cowrie
    from cowrie_vllm_adapters._plugin import register_vllm_plugin
    from cowrie_vllm_adapters.cowrie_kv_connector import (
        read_connector_stats,
        reset_connector_stats,
    )
    from vllm import LLM, SamplingParams
    from vllm.config import KVTransferConfig

    # Always reset per-phase stats files before engine start (shards stay intact for restore!)
    reset_connector_stats(state_dir)
    register_vllm_plugin()

    kv_cfg = KVTransferConfig(
        kv_connector="CowrieKVConnectorV1",
        kv_role="kv_both",
        kv_connector_extra_config={
            "cowrie_state_dir": str(state_dir),
            "cowrie_max_inflight_bytes": int(cfg["cowrie_max_inflight_bytes"]),
            "cowrie_max_cache_bytes": int(cfg["cowrie_max_cache_bytes"]),
        },
    )

    llm_kwargs = dict(
        model=cfg["model_id"],
        dtype=cfg["dtype"],
        max_model_len=int(cfg["max_model_len"]),
        gpu_memory_utilization=float(cfg["gpu_memory_utilization"]),
        enable_prefix_caching=True,
        enforce_eager=bool(cfg["enforce_eager"]),
        kv_transfer_config=kv_cfg,
    )
    if cfg.get("quantization"):
        llm_kwargs["quantization"] = cfg["quantization"]

    t_init0 = time.perf_counter()
    llm = LLM(**llm_kwargs)
    engine_init_s = round(time.perf_counter() - t_init0, 2)

    # Verify BPE prefix alignment for the new-suffix prompt before running
    tok = llm.get_tokenizer()
    block_size = 16
    try:
        block_size = int(llm.llm_engine.vllm_config.cache_config.block_size)
    except Exception:
        pass
    seed_ids = tok.encode(prompts["prompt_prefix_seed"])
    new_suffix_ids = tok.encode(prompts["prompt_new_suffix"])
    aligned_n = ((len(seed_ids) - 1) // block_size) * block_size
    assert aligned_n > 0 and new_suffix_ids[:aligned_n] == seed_ids[:aligned_n], (
        f"BPE boundary mismatch on prefix_suffix corpus (aligned_n={aligned_n})!"
    )

    sp_warmup = SamplingParams(temperature=0.0, max_tokens=4)
    sp_ttft = SamplingParams(temperature=0.0, max_tokens=1)
    decode_n = int(cfg["decode_max_tokens"])
    sp_decode = SamplingParams(temperature=0.0, ignore_eos=True, max_tokens=decode_n)

    # Short warmup (< block_size tokens) so CUDA kernels are warm without polluting prefix cache
    llm.generate(["Warmup."], sp_warmup, use_tqdm=False)

    # 1. TTFT probe (disjoint prefix A, max_tokens=1)
    t0 = time.perf_counter()
    out_ttft = llm.generate([prompts["prompt_ttft"]], sp_ttft, use_tqdm=False)[0]
    ttft_ms = (time.perf_counter() - t0) * 1000.0
    m_ttft = _extract_metrics_optional(out_ttft)

    # 2. Decode & greedy parity probe (disjoint prefix B, exact decode_n tokens)
    t1 = time.perf_counter()
    out_dec = llm.generate([prompts["prompt_decode"]], sp_decode, use_tqdm=False)[0]
    decode_e2e_ms = (time.perf_counter() - t1) * 1000.0
    m_dec = _extract_metrics_optional(out_dec)

    # 3. Prefix seed (store) OR brand-new suffix reuse (restore) on disjoint prefix C
    if args.phase == "store":
        t2 = time.perf_counter()
        out_p3 = llm.generate([prompts["prompt_prefix_seed"]], sp_ttft, use_tqdm=False)[0]
        p3_ms = (time.perf_counter() - t2) * 1000.0
    else:
        t2 = time.perf_counter()
        out_p3 = llm.generate([prompts["prompt_new_suffix"]], sp_decode, use_tqdm=False)[0]
        p3_ms = (time.perf_counter() - t2) * 1000.0
    m_p3 = _extract_metrics_optional(out_p3)

    # Cleanly shut down EngineCore (triggers connector.shutdown() -> io_pool.shutdown(wait=True))
    _shutdown_llm(llm)

    # Wait up to 30s for any asynchronous background StatePage writes to finish flushing
    if args.phase == "store":
        deadline = time.perf_counter() + 30.0
        while time.perf_counter() < deadline:
            cur_stats = read_connector_stats(state_dir)["totals"]
            if (
                cur_stats.get("inflight_saves", 0) == 0
                and (
                    cur_stats.get("saves_completed", 0) >= 3
                    or cur_stats.get("saves_failed", 0) > 0
                    or cur_stats.get("saves_skipped_inflight_cap", 0) > 0
                )
            ):
                break
            time.sleep(0.2)

    connector_stats = read_connector_stats(state_dir)
    shards = sorted(state_dir.glob("*.state.cowrie"))
    verified_count = 0
    all_hmac = bool(shards)
    for sp in shards:
        info = cowrie.verify_statepage_file(
            str(sp),
            expected_runtime_id="vllm-v1",
            verify_hash=True,
        )
        verified_count += 1
        all_hmac = all_hmac and bool(info.get("hmac_verified", False))

    cached_vals = [
        x for x in (m_ttft["num_cached_tokens"], m_dec["num_cached_tokens"], m_p3["num_cached_tokens"])
        if x is not None
    ]
    vllm_cached_total = sum(cached_vals) if len(cached_vals) == 3 else None

    result = {
        "phase": args.phase,
        "engine_init_s": engine_init_s,
        "ttft_ms": round(ttft_ms, 2),
        "ttft_prompt_tokens": len(out_ttft.prompt_token_ids),
        "ttft_vllm_metrics": m_ttft,
        "decode_e2e_ms": round(decode_e2e_ms, 2),
        "decode_prompt_tokens": len(out_dec.prompt_token_ids),
        "decode_output_tokens": len(out_dec.outputs[0].token_ids),
        "decode_token_ids": list(out_dec.outputs[0].token_ids),
        "decode_text_preview": out_dec.outputs[0].text[:200],
        "decode_vllm_metrics": m_dec,
        "new_suffix_e2e_ms": round(p3_ms, 2),
        "new_suffix_prompt_tokens": len(out_p3.prompt_token_ids),
        "new_suffix_output_tokens": len(out_p3.outputs[0].token_ids),
        "new_suffix_vllm_metrics": m_p3,
        "vllm_num_cached_tokens_total": vllm_cached_total,
        "state_shards": [s.name for s in shards],
        "verified_shards_count": verified_count,
        "state_bytes_total": sum(s.stat().st_size for s in shards),
        "all_shards_hmac_verified": all_hmac,
        "connector_stats": connector_stats,
    }
    Path(args.result_json).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"PHASE {args.phase.upper()} COMPLETE -> {args.result_json}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

WORKER_SCRIPT_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
print(f"Wrote {WORKER_SCRIPT_PATH} ({len(WORKER_SOURCE)} bytes)")

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

# Start from a clean STATE_DIR so Phase 1 is 100% cold
if STATE_DIR.exists():
    shutil.rmtree(STATE_DIR)
STATE_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)

cfg_file = Path("/tmp/cowrie_exp04_cfg.json")
prompts_file = Path("/tmp/cowrie_exp04_prompts.json")
store_out_file = Path("/tmp/cowrie_exp04_store.json")
restore_out_file = Path("/tmp/cowrie_exp04_restore.json")

cfg_file.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
prompts_file.write_text(json.dumps(PROMPTS, indent=2), encoding="utf-8")

worker_env = os.environ.copy()
worker_env["COWRIE_HMAC_KEY"] = COWRIE_HMAC_KEY_HEX


def _run_phase(phase: str, out_file: Path) -> dict:
    if out_file.exists():
        out_file.unlink()
    cmd = [
        sys.executable,
        str(WORKER_SCRIPT_PATH),
        "--phase", phase,
        "--config-json", str(cfg_file),
        "--prompts-json", str(prompts_file),
        "--result-json", str(out_file),
    ]
    print(f"\n========== LAUNCHING OS SUBPROCESS: PHASE {phase.upper()} ==========", flush=True)
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, env=worker_env)
    wall_s = round(time.perf_counter() - t0, 2)
    if proc.returncode != 0:
        raise RuntimeError(f"Phase {phase} subprocess exited with rc={proc.returncode}")
    assert out_file.is_file(), f"Phase {phase} did not produce {out_file}"
    data = json.loads(out_file.read_text(encoding="utf-8"))
    data["subprocess_wall_s"] = wall_s
    return data


STORE_RES = _run_phase("store", store_out_file)
RESTORE_RES = _run_phase("restore", restore_out_file)

# Fail loudly if SupportsHMA failed, connector was silently unused, or greedy parity broke!
assert_connector_was_used(STORE_RES, RESTORE_RES, HMA_PROOF)
print("\nFAIL-LOUD VERIFICATION PASSED: CowrieKVConnectorV1 + SupportsHMA genuinely executed across restart.")

In [ ]:
import json

SUMMARY = compute_benchmark_summary(STORE_RES, RESTORE_RES, HMA_PROOF, GPU_INFO)
RECEIPT_PATH.write_text(json.dumps(SUMMARY, indent=2), encoding="utf-8")

m = SUMMARY["metrics"]
st = SUMMARY["connector_counters"]["phase_1_store"]["totals"]
rt = SUMMARY["connector_counters"]["phase_2_restore"]["totals"]

print("=" * 78)
print("EXP-04 (CLOUD-LANE-01) — LIVE vLLM 0.15+ CowrieKVConnectorV1 RESULTS")
print("=" * 78)
print(f"GPU / Compute Cap       : {GPU_INFO['gpu_name']} ({GPU_INFO['vram_gib']} GiB, sm_{GPU_INFO['compute_capability']})")
print(f"Model / Dtype / Eager   : {CONFIG['model_id']} | {CONFIG['dtype']} | enforce_eager={CONFIG['enforce_eager']}")
print(f"vLLM / Cowrie SHA       : vLLM {HMA_PROOF['vllm_version']} | cowrie {HMA_PROOF['cowrie_git_sha']}")
print(f"SupportsHMA (Runtime)   : {HMA_PROOF['issubclass_supports_hma']} (native={HMA_PROOF['supports_hma_native_import']}, live_all={rt['all_supports_hma']})")
print(f"KV Tensor Layout        : {rt['kv_layouts']}")
print("-" * 78)
print(f"TTFT (Cold Prefill)     : {m['cold_ttft_ms']:.2f} ms ({m['prompt_tokens_ttft']} prompt tok)")
print(f"TTFT (Warm Cowrie)      : {m['warm_cowrie_ttft_ms']:.2f} ms (speedup: {m['ttft_speedup_x']:.2f}x)")
print(f"Decode E2E (Cold)       : {m['cold_decode_e2e_ms']:.2f} ms -> {m['cold_e2e_tokens_per_s']:.2f} tok/s ({m['decode_output_tokens']} out tok)")
print(f"Decode E2E (Warm Cowrie): {m['warm_decode_e2e_ms']:.2f} ms -> {m['warm_e2e_tokens_per_s']:.2f} tok/s")
print(f"Decode-Only (Warm est.) : {m['warm_decode_only_tokens_per_s']:.2f} tok/s")
print(f"New-Suffix Reuse (Warm) : {m['new_suffix_warm_e2e_ms']:.2f} ms -> {m['new_suffix_warm_e2e_tokens_per_s']:.2f} tok/s")
print("-" * 78)
print(f"KV Cache Hit Rate (Disk): {m['kv_cache_hit_rate_loaded_pct']:.2f}% ({rt['loaded_tokens']}/{m['restore_prompt_tokens_total']} tok)")
print(f"KV Cache Hit Rate (Sch) : {m['kv_cache_hit_rate_scheduler_pct']:.2f}% ({rt['scheduler_matched_tokens']}/{m['restore_prompt_tokens_total']} tok)")
print(f"Request Hit Rate        : {m['kv_cache_request_hit_rate_pct']:.2f}% ({rt['scheduler_hits']}/3 prompts)")
print(f"vLLM num_cached_tokens  : {m['vllm_num_cached_tokens_total']}")
print(f"Greedy Token Parity     : {m['greedy_token_parity']} | HMAC Verified: {m['hmac_sha256_verified']} ({m['state_shards_count']} shards, {m['state_shards_total_bytes'] / (1024**2):.1f} MiB)")
print("-" * 78)
print("Connector Counters — Phase 1 (Store):")
print(json.dumps(st, indent=2))
print("Connector Counters — Phase 2 (Restore):")
print(json.dumps(rt, indent=2))
print("=" * 78)
print(f"VERDICT: {SUMMARY['verdict']}")
print(f"Saved JSON receipt to: {RECEIPT_PATH}")

## 4. Colab Runtime Teardown (Prevent Idle CU Burn)

Download `/tmp/cowrie_exp04_receipt.json` into `receipts/` before unassigning the runtime. Set `EXP04_AUTO_UNASSIGN=1` (or flip `AUTO_UNASSIGN_COLAB_RUNTIME = True` in Cell 1) for unattended runs.

In [ ]:
if AUTO_UNASSIGN_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("Unassigning Colab runtime to stop CU burn...")
        runtime.unassign()
    except ImportError:
        print("Not running inside Google Colab; skipping runtime.unassign().")
else:
    print(
        f"Receipt ready at {RECEIPT_PATH}. "
        "Download it, then disconnect/delete the Colab runtime (or call google.colab.runtime.unassign()) "
        "to avoid burning idle Compute Units."
    )